# Foodpanda-Style AI Support Agent

A mock food-delivery support system built with **LangChain**, **LangGraph**, **Google Gemini**, **SQLite**, **ChromaDB (RAG)** and **Streamlit**.

The LLM understands and phrases; business rules are enforced in code, never by the LLM.

# Phase 1: Project Setup (Colab)

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Project folder
import os

project_path = '/content/drive/MyDrive/foodpanda-ai-agent'
os.makedirs(project_path, exist_ok=True)
os.chdir(project_path)
print("Working directory:", os.getcwd())

In [ ]:
# Install packages
!pip install langchain langgraph langchain-google-genai python-dotenv -q

In [ ]:
# Test
import langchain
import langgraph
print("Setup successful! Working in:", os.getcwd())

# Phase 2: SQLite Database & Order Data

In [ ]:
# Create table
import sqlite3

DB_PATH = "foodpanda.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id TEXT PRIMARY KEY,
    customer_name TEXT,
    restaurant TEXT,
    items TEXT,
    order_status TEXT,
    delivery_address TEXT,
    rider_name TEXT,
    rider_status TEXT,
    payment_status TEXT,
    order_time TEXT,
    total_amount REAL
)
""")

conn.commit()
print("Table created (or already existed).")

In [ ]:
# Sample data
sample_orders = [
    ("ORD1001", "Fatima Khan", "KFC", "2x Zinger Burger, 1x Fries",
     "preparing", "House 12, Model Town, Lahore", "Ali Raza", "not_assigned",
     "paid", "2026-09-15 12:30:00", 1450.0),

    ("ORD1002", "Ahmed Ali", "Pizza Hut", "1x Large Pepperoni Pizza",
     "out_for_delivery", "Flat 3B, DHA Phase 5, Lahore", "Bilal Hussain", "on_the_way",
     "paid", "2026-09-15 11:50:00", 1899.0),

    ("ORD1003", "Sara Malik", "McDonald's", "1x Big Mac Meal, 1x McFlurry",
     "delivered", "45 Gulberg III, Lahore", "Usman Tariq", "delivered",
     "paid", "2026-09-15 10:15:00", 990.0),

    ("ORD1004", "Zainab Iqbal", "Subway", "2x Chicken Tikka Sub",
     "placed", "Street 7, Johar Town, Lahore", None, "not_assigned",
     "cash_on_delivery", "2026-09-15 13:05:00", 1200.0),
]

cursor.executemany("""
INSERT OR IGNORE INTO orders VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", sample_orders)

conn.commit()
print("Orders inserted.")

In [ ]:
# Reset helper - defined early so later cells can always start from a known state.
# (Phase 16 below replaces this with an expanded version that also resets
# restaurants, menu items, and reviews once those tables exist.)
def reset_database():
    """Wipes and re-seeds the orders table with the original sample data."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM orders")
    cursor.executemany("INSERT OR IGNORE INTO orders VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", sample_orders)
    conn.commit()
    conn.close()
    print("Database reset to original sample data.")


In [ ]:
# Test
cursor.execute("SELECT order_id, customer_name, order_status FROM orders")
for row in cursor.fetchall():
    print(row)

print(os.path.exists("foodpanda.db"))

# Phase 3: Backend/Database Functions — db_functions.py

In [ ]:
%%writefile db_functions.py

import sqlite3

DB_PATH = "foodpanda.db"

def get_connection():
    return sqlite3.connect(DB_PATH)


def get_order(order_id):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM orders WHERE order_id = ?", (order_id,))
    row = cursor.fetchone()
    conn.close()

    if row is None:
        return None

    columns = ["order_id", "customer_name", "restaurant", "items", "order_status",
               "delivery_address", "rider_name", "rider_status", "payment_status",
               "order_time", "total_amount"]
    return dict(zip(columns, row))


def get_order_status(order_id):
    order = get_order(order_id)
    return order["order_status"] if order else None


def update_delivery_address(order_id, new_address):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE orders SET delivery_address = ? WHERE order_id = ?",
        (new_address, order_id)
    )
    conn.commit()
    updated = cursor.rowcount > 0
    conn.close()
    return updated


def update_order_status(order_id, new_status):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE orders SET order_status = ? WHERE order_id = ?",
        (new_status, order_id)
    )
    conn.commit()
    updated = cursor.rowcount > 0
    conn.close()
    return updated


def get_rider_status(order_id):
    order = get_order(order_id)
    return order["rider_status"] if order else None

In [ ]:
# Import
import db_functions
import importlib
importlib.reload(db_functions)

In [ ]:
# Test
order = db_functions.get_order("ORD1001")
print("Order ORD1001:", order)

status = db_functions.get_order_status("ORD1002")
print("Status of ORD1002:", status)

missing = db_functions.get_order("ORD9999")
print("Missing order result:", missing)

success = db_functions.update_delivery_address("ORD1004", "New Address, DHA Phase 6, Lahore")
print("Address update success:", success)
print("Updated order:", db_functions.get_order("ORD1004"))

# Phase 4: LangChain Tools — tools.py

In [ ]:
%%writefile tools.py

from langchain_core.tools import tool
import db_functions

CANCELLABLE_STATUSES = ["placed", "preparing"]
ADDRESS_CHANGEABLE_STATUSES = ["placed", "preparing"]


@tool
def check_order(order_id: str) -> str:
    """Look up full details for an order by its order ID."""
    order = db_functions.get_order(order_id)
    if order is None:
        return f"No order found with ID {order_id}."
    return str(order)


@tool
def get_order_status(order_id: str) -> str:
    """Get just the current status of an order by its order ID."""
    status = db_functions.get_order_status(order_id)
    if status is None:
        return f"No order found with ID {order_id}."
    return f"Order {order_id} status: {status}"


@tool
def get_rider_status(order_id: str) -> str:
    """Get the current rider status for an order by its order ID."""
    rider_status = db_functions.get_rider_status(order_id)
    if rider_status is None:
        return f"No order found with ID {order_id}."
    return f"Rider status for order {order_id}: {rider_status}"


@tool
def change_address(order_id: str, new_address: str) -> str:
    """Attempt to change the delivery address of an order. Only allowed if the
    order status is 'placed' or 'preparing'."""
    order = db_functions.get_order(order_id)
    if order is None:
        return f"No order found with ID {order_id}."

    if order["order_status"] not in ADDRESS_CHANGEABLE_STATUSES:
        return (f"Address cannot be changed. Order {order_id} is already "
                f"'{order['order_status']}', which is too late to update the address.")

    success = db_functions.update_delivery_address(order_id, new_address)
    if success:
        return f"Address for order {order_id} updated to: {new_address}"
    return f"Failed to update address for order {order_id}."


@tool
def cancel_order(order_id: str) -> str:
    """Attempt to cancel an order. Only allowed if the order status is 'placed'
    or 'preparing'."""
    order = db_functions.get_order(order_id)
    if order is None:
        return f"No order found with ID {order_id}."

    if order["order_status"] not in CANCELLABLE_STATUSES:
        return (f"Order {order_id} cannot be cancelled because it is already "
                f"'{order['order_status']}'.")

    success = db_functions.update_order_status(order_id, "cancelled")
    if success:
        return f"Order {order_id} has been cancelled successfully."
    return f"Failed to cancel order {order_id}."


@tool
def check_refund(order_id: str, previous_status: str = "") -> str:
    """Check the refund status for a cancelled order. previous_status is the order's
    status BEFORE it was cancelled ('placed' or 'preparing'); it decides the refund amount."""
    order = db_functions.get_order(order_id)
    if order is None:
        return f"No order found with ID {order_id}."

    if order["order_status"] != "cancelled":
        return f"Order {order_id} is not cancelled, so no refund applies."

    if order["payment_status"] != "paid":
        return (f"Order {order_id} was cancelled — no refund needed "
                f"(payment was {order['payment_status']}).")

    if previous_status == "placed":
        return (f"Order {order_id} was paid and cancelled before preparation started — "
                f"a full refund has been initiated (3-5 business days).")
    if previous_status == "preparing":
        return (f"Order {order_id} was paid and cancelled while being prepared — "
                f"a 50% refund has been initiated, per the refund policy.")
    return f"Order {order_id} was paid and cancelled — a refund has been initiated per the refund policy."


In [ ]:
# Import
import tools
import importlib
importlib.reload(tools)

In [ ]:
# Ensure a clean, known state before this demo (ORD1001 must still be
# "preparing" here so the cancel/refund/address-change calls below behave
# as the comments describe).
reset_database()


In [ ]:
# Test
print(tools.check_order.invoke({"order_id": "ORD1001"}))
print(tools.get_order_status.invoke({"order_id": "ORD1002"}))
print(tools.cancel_order.invoke({"order_id": "ORD1002"}))
print(tools.cancel_order.invoke({"order_id": "ORD1001"}))
print(tools.check_refund.invoke({"order_id": "ORD1001"}))
print(tools.change_address.invoke({"order_id": "ORD1001", "new_address": "New Street, Lahore"}))

# Phase 5: Connecting the LLM (Gemini)

In [ ]:
# Install
!pip install langchain-google-genai -q

In [ ]:
# Enter your Gemini API key
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google Gemini API key: ")

In [ ]:
# List available models — always check this if you get a "model not found" error
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

In [ ]:
# Connect — use a "lite" model for a higher free-tier daily quota
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0
)

In [ ]:
# Helper — Gemini can return structured (list-of-blocks) content instead of plain text
def get_text(response):
    if isinstance(response.content, str):
        return response.content
    return "".join(
        block.get("text", "")
        for block in response.content
        if isinstance(block, dict)
    )

In [ ]:
# Test
response = llm.invoke("In one short sentence, what is the capital of France?")
print(get_text(response))

response2 = llm.invoke("Reply with only the word: CONNECTED")
print(get_text(response2))

# Phase 6: Manual Tool-Calling Workflow

In [ ]:
import tools as tools_module

all_tools = [
    tools_module.check_order,
    tools_module.get_order_status,
    tools_module.get_rider_status,
    tools_module.change_address,
    tools_module.cancel_order,
    tools_module.check_refund,
]

llm_with_tools = llm.bind_tools(all_tools)

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

tool_lookup = {t.name: t for t in all_tools}

def run_tools_and_respond(messages, ai_response):
    messages = messages + [ai_response]

    for tool_call in ai_response.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_id = tool_call["id"]

        selected_tool = tool_lookup[tool_name]
        result = selected_tool.invoke(tool_args)

        messages.append(ToolMessage(content=str(result), tool_call_id=tool_id))

    final_response = llm_with_tools.invoke(messages)
    return final_response, messages

In [ ]:
# Test
messages = [HumanMessage(content="What's the status of order ORD1002?")]
ai_response = llm_with_tools.invoke(messages)
final_response, _ = run_tools_and_respond(messages, ai_response)
print(get_text(final_response))

# Phase 8: Real State — state.py

In [ ]:
%%writefile state.py

from typing import TypedDict, Optional


class SupportState(TypedDict):
    user_message: str
    history: Optional[str]   # recent chat turns, so the agent can follow up
    user_id: Optional[str]

    order_id: Optional[str]
    order_status: Optional[str]
    current_address: Optional[str]

    action: Optional[str]
    new_address: Optional[str]

    rider_status: Optional[str]

    result: Optional[str]
    response: Optional[str]

In [ ]:
# Import
import state
import importlib
importlib.reload(state)
from state import SupportState

# Phase 9 & 11: Nodes — nodes.py

In [ ]:
%%writefile nodes.py

import json
from langchain_core.messages import HumanMessage
from state import SupportState
import db_functions


def get_text(response):
    """Handles both plain-string and structured (list-of-blocks) Gemini responses."""
    if isinstance(response.content, str):
        return response.content
    return "".join(
        block.get("text", "")
        for block in response.content
        if isinstance(block, dict)
    )


def make_understand_request_node(llm):
    """Returns a node function that uses the LLM to figure out the user's
    intent (action), which order they mean (order_id), and — for address
    changes — the new address they want."""

    def understand_request(state: SupportState) -> dict:
        prompt = f"""You are analyzing a customer support message for a food delivery app.

Conversation so far (may be empty):
{state.get('history') or '(none)'}

Latest customer message: "{state['user_message']}"

IMPORTANT: the latest message is often an answer to something the assistant just asked
(for example only an order ID, or only a new address). Use the conversation so far to work
out what the customer really wants, and carry over the order_id and new_address from earlier
messages when the latest message does not repeat them. If the latest message is only an order ID
or only an address, keep the earlier intent (cancel / change_address / check_status) - do NOT
answer with "other" or "policy_question".
If the customer asks for two things in one message (for example "cancel it, and if you can't,
change the address"), set "action" to the FIRST thing they ask for.

Figure out three things:
1. "action": one of exactly these values: "cancel", "change_address", "check_status", "other"
2. "order_id": the order ID mentioned in the message (like "ORD1002"), or null if none is mentioned
3. "new_address": if the user gave a new delivery address, put it here as a string; otherwise null

Reply with ONLY a JSON object, nothing else, like this:
{{"action": "change_address", "order_id": "ORD1002", "new_address": "House 5, Gulberg, Lahore"}}
"""
        response = llm.invoke([HumanMessage(content=prompt)])
        text = get_text(response).strip()

        if text.startswith("```"):
            text = text.strip("`")
            text = text.replace("json", "", 1).strip()

        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            parsed = {"action": "other", "order_id": None, "new_address": None}

        return {
            "action": parsed.get("action"),
            "order_id": parsed.get("order_id"),
            "new_address": parsed.get("new_address"),
        }

    return understand_request


def find_order_node(state: SupportState) -> dict:
    """Looks up the order in the database. Pure database lookup — no LLM."""
    order_id = state.get("order_id")

    if not order_id:
        return {"result": "no_order_id"}

    order = db_functions.get_order(order_id)

    if order is None:
        return {"result": "order_not_found"}

    return {
        "order_status": order["order_status"],
        "current_address": order["delivery_address"],
        "rider_status": order["rider_status"],
    }


# Business rules — plain Python, no LLM involved in deciding these
CANCELLABLE_STATUSES = ["placed", "preparing"]
ADDRESS_CHANGEABLE_STATUSES = ["placed", "preparing"]


def decide_action(state: SupportState) -> dict:
    """Pure business-rule decision — NO LLM involved. Looks at the real
    order_status and the requested action, and decides the outcome."""
    action = state.get("action")
    order_status = state.get("order_status")

    if state.get("result") in ("no_order_id", "order_not_found"):
        return {}  # already handled — nothing to decide

    if order_status == "cancelled" and action in ("cancel", "change_address"):
        return {"result": "already_cancelled"}

    if action == "cancel":
        if order_status in CANCELLABLE_STATUSES:
            success = db_functions.update_order_status(state["order_id"], "cancelled")
            return {"result": "cancelled" if success else "cancel_failed"}
        return {"result": "cancel_blocked"}

    if action == "change_address":
        if order_status not in ADDRESS_CHANGEABLE_STATUSES:
            return {"result": "address_blocked"}
        new_address = state.get("new_address")
        if not new_address:
            return {"result": "address_missing"}
        success = db_functions.update_delivery_address(state["order_id"], new_address)
        return {"result": "address_updated" if success else "address_update_failed"}

    if action == "check_status":
        return {"result": "status_checked"}

    return {"result": "unrecognized_action"}


RESULT_MEANINGS = {
    "no_order_id": "The customer has not given an order ID yet. Ask for it.",
    "order_not_found": "No order exists with that ID. Ask them to double-check the ID.",
    "cancelled": "The order was successfully cancelled. Refund details (if any) follow in the result text.",
    "cancel_blocked": "The order could NOT be cancelled because it is already out for delivery or delivered. Orders can only be cancelled while 'placed' or 'preparing'.",
    "cancel_failed": "Something went wrong while cancelling. Apologise and suggest trying again.",
    "address_updated": "The delivery address was successfully updated to the new address.",
    "address_missing": "The customer wants to change the address but has not said the new address yet. Ask for it.",
    "address_blocked": "The address could NOT be changed because the order is already out for delivery or delivered. Addresses can only be changed while the order is 'placed' or 'preparing'.",
    "address_update_failed": "Something went wrong while updating the address. Apologise and suggest trying again.",
    "status_checked": "Tell the customer the current order status, and the rider information if it is given.",
    "already_cancelled": "This order was ALREADY cancelled earlier, so it cannot be changed or cancelled again, and a cancelled order cannot be reactivated. If they still want food, they need to place a new order.",
}


def explain_result(result):
    """Turns a result code into a plain-English meaning, so the LLM never has to guess."""
    result = result or ""
    for key in sorted(RESULT_MEANINGS, key=len, reverse=True):
        if result.startswith(key):
            return RESULT_MEANINGS[key]
    return "No special outcome."


def make_respond_node(llm):
    """Returns a node function that turns the final state into a friendly,
    natural-language response — but only DESCRIBES the result, never decides it."""

    def respond(state: SupportState) -> dict:
        prompt = f"""You are a friendly food delivery support assistant.

Here is what happened when handling the customer's request:
- Customer said: "{state['user_message']}"
- Order ID: {state.get('order_id')}
- Order status: {state.get('order_status')}
- Result: {state.get('result')}
- What this result means: {explain_result(state.get('result'))}
- Rider info: {state.get('rider_status')}

Conversation so far (may be empty):
{state.get('history') or '(none)'}

Write a short, friendly response to the customer explaining the outcome.
If the result is "address_missing", politely ask them what new address they'd like.
If the result is "cancel_blocked" and the customer also asked to change the address instead, tell them the address cannot be changed at this stage either, because the same rule applies (only orders that are "placed" or "preparing" can be changed).
Base your answer ONLY on the result and its meaning above. Never invent a reason that is not stated there.
If there is earlier conversation, do NOT start with a greeting like "Hi there" again.
Do not invent any information that isn't given above.
"""
        response = llm.invoke([HumanMessage(content=prompt)])
        return {"response": get_text(response)}

    return respond


import tools as tools_module

def gather_extra_info(state: SupportState) -> dict:
    """Deterministically calls LangChain tools based on the already-decided
    result — NOT based on LLM judgment. This is where LangChain tools and
    LangGraph flow control meet (Phase 12)."""
    result = state.get("result")
    order_id = state.get("order_id")
    updates = {}

    if result == "cancelled":
        refund_info = tools_module.check_refund.invoke(
            {"order_id": order_id, "previous_status": state.get("order_status") or ""}
        )
        updates["result"] = f"cancelled. {refund_info}"

    elif result == "status_checked":
        rider_info = tools_module.get_rider_status.invoke({"order_id": order_id})
        updates["rider_status"] = rider_info

    return updates

In [ ]:
# Import
import nodes
import importlib
importlib.reload(nodes)

# Phase 10, 11 & 12: Edges & Conditional Edges — graph.py

In [ ]:
%%writefile graph.py

from langgraph.graph import StateGraph, END
from state import SupportState
import nodes


def route_after_find_order(state: SupportState) -> str:
    """Conditional edge function. Looks at state, returns the NAME of the
    next node to go to. Runs after find_order."""
    if state.get("result") in ("no_order_id", "order_not_found"):
        return "respond"
    return "decide_action"


def build_graph(llm):
    understand_request = nodes.make_understand_request_node(llm)
    respond = nodes.make_respond_node(llm)

    graph_builder = StateGraph(SupportState)

    graph_builder.add_node("understand_request", understand_request)
    graph_builder.add_node("find_order", nodes.find_order_node)
    graph_builder.add_node("decide_action", nodes.decide_action)
    graph_builder.add_node("gather_extra_info", nodes.gather_extra_info)
    graph_builder.add_node("respond", respond)

    graph_builder.set_entry_point("understand_request")
    graph_builder.add_edge("understand_request", "find_order")

    graph_builder.add_conditional_edges(
        "find_order",
        route_after_find_order,
        {
            "respond": "respond",
            "decide_action": "decide_action",
        }
    )

    graph_builder.add_edge("decide_action", "gather_extra_info")
    graph_builder.add_edge("gather_extra_info", "respond")
    graph_builder.add_edge("respond", END)

    return graph_builder.compile()

In [ ]:
# Rebuild the app — force a fresh import of every module to avoid stale-cache issues
import sys
for mod_name in ["db_functions", "tools", "state", "nodes", "graph"]:
    if mod_name in sys.modules:
        del sys.modules[mod_name]

import db_functions
import tools
import state
from state import SupportState
import nodes
import graph as graph_module

app = graph_module.build_graph(llm)
print("Graph built successfully.")

In [ ]:
# Ensure a clean, known state before these demos - earlier cells (e.g. the
# Phase 4 tool test) may already have changed ORD1001 / ORD1002's status,
# which would otherwise make this "allowed cancel" demo show as blocked.
reset_database()


In [ ]:
# Test — blocked cancel (out for delivery)
initial_state = {
    "user_message": "Please cancel my order ORD1002", "user_id": "U001",
    "order_id": None, "order_status": None, "current_address": None,
    "action": None, "new_address": None, "rider_status": None,
    "result": None, "response": None,
}
final_state = app.invoke(initial_state)
print("Result:", final_state["result"])
print("Response:", final_state["response"])

In [ ]:
# Test — allowed cancel + auto refund check (Phase 12)
initial_state2 = {
    "user_message": "I want to cancel order ORD1001", "user_id": "U001",
    "order_id": None, "order_status": None, "current_address": None,
    "action": None, "new_address": None, "rider_status": None,
    "result": None, "response": None,
}
final_state2 = app.invoke(initial_state2)
print("Result:", final_state2["result"])
print("Response:", final_state2["response"])

In [ ]:
# Test — address change with an address given in the same message
initial_state3 = {
    "user_message": "Change the delivery address for ORD1004 to House 9, Gulberg III, Lahore",
    "user_id": "U001", "order_id": None, "order_status": None,
    "current_address": None, "action": None, "new_address": None,
    "rider_status": None, "result": None, "response": None,
}
final_state3 = app.invoke(initial_state3)
print("Result:", final_state3["result"])
print("Response:", final_state3["response"])

In [ ]:
# Test — status check + rider info (Phase 12)
initial_state4 = {
    "user_message": "Where is my order ORD1002?", "user_id": "U001",
    "order_id": None, "order_status": None, "current_address": None,
    "action": None, "new_address": None, "rider_status": None,
    "result": None, "response": None,
}
final_state4 = app.invoke(initial_state4)
print("Result:", final_state4["result"])
print("Rider status:", final_state4["rider_status"])
print("Response:", final_state4["response"])

# Phase 13: AI Support Chat Interface

In [ ]:
def make_initial_state(user_message: str, history: str = "", user_id: str = "U001"):
    return {
        "user_message": user_message,
        "history": history,
        "user_id": user_id,
        "order_id": None,
        "order_status": None,
        "current_address": None,
        "action": None,
        "new_address": None,
        "rider_status": None,
        "result": None,
        "response": None,
    }

In [ ]:
def ask_support_agent(user_message: str):
    """Runs one message through the graph and prints the response."""
    initial_state = make_initial_state(user_message)
    final_state = app.invoke(initial_state)
    print("You:", user_message)
    print("Agent:", final_state["response"])
    print()
    return final_state

In [ ]:
def chat_loop():
    print("Foodpanda Support Chat — type 'quit' to stop.\n")
    turns = []   # (role, message) pairs - this is the conversation memory
    while True:
        user_message = input("You: ")
        if user_message.strip().lower() == "quit":
            print("Chat ended.")
            break
        history = "\n".join(f"{role}: {msg}" for role, msg in turns[-8:])
        final_state = app.invoke(make_initial_state(user_message, history))
        print("Agent:", final_state["response"])
        print()
        turns.append(("user", user_message))
        turns.append(("assistant", final_state["response"]))


In [ ]:
# Test
ask_support_agent("What's the status of order ORD1002?")

In [ ]:
# Interactive test (run in its own cell)
# chat_loop()

# Phase 14: Testing Different Scenarios

In [ ]:
# Reset to a known, clean state before testing
reset_database()

In [ ]:
test_cases = [
    {"label": "Cancel — allowed (preparing)", "message": "Please cancel my order ORD1001", "expected_result": "cancelled"},
    {"label": "Cancel — blocked (out for delivery)", "message": "I want to cancel order ORD1002", "expected_result": "cancel_blocked"},
    {"label": "Cancel — blocked (already delivered)", "message": "Cancel ORD1003 please", "expected_result": "cancel_blocked"},
    {"label": "Check status — out for delivery", "message": "Where is my order ORD1002?", "expected_result": "status_checked"},
    {"label": "Change address — allowed (placed)", "message": "Change delivery address for ORD1004 to Street 9, Bahria Town, Lahore", "expected_result": "address_updated"},
    {"label": "Change address — missing new address", "message": "I want to update my address for ORD1004", "expected_result": "address_missing"},
    {"label": "Order ID missing entirely", "message": "I want to cancel my order", "expected_result": "no_order_id"},
    {"label": "Order ID not found", "message": "Cancel order ORD9999", "expected_result": "order_not_found"},
    {"label": "Unrelated / unclear request", "message": "What toppings does Pizza Hut have?", "expected_result": "unrecognized_action"},
]

In [ ]:
def run_test_suite(test_cases):
    passed = 0
    failed = 0

    for case in test_cases:
        state = app.invoke(make_initial_state(case["message"]))
        actual = state.get("result", "")
        success = actual.startswith(case["expected_result"])

        status = "PASS" if success else "FAIL"
        if success:
            passed += 1
        else:
            failed += 1

        print(f"[{status}] {case['label']}")
        print(f"    Message : {case['message']}")
        print(f"    Expected: {case['expected_result']}")
        print(f"    Actual  : {actual}")
        print(f"    Response: {state['response']}")
        print()

    print(f"Summary: {passed} passed, {failed} failed out of {len(test_cases)}")

run_test_suite(test_cases)

In [ ]:
# Verify all files are in one place
import os
print(os.listdir("."))

In [ ]:
# Final smoke test
reset_database()
run_test_suite(test_cases)

# Phase 16: Expanding the Database — Restaurants, Menu, Cart, Reviews

In [ ]:
# Create the new tables
import sqlite3

DB_PATH = "foodpanda.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS restaurants (
    restaurant_id TEXT PRIMARY KEY,
    name TEXT,
    cuisine TEXT,
    rating REAL,
    delivery_time_mins INTEGER,
    address TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS menu_items (
    item_id TEXT PRIMARY KEY,
    restaurant_id TEXT,
    name TEXT,
    description TEXT,
    price REAL,
    category TEXT,
    FOREIGN KEY (restaurant_id) REFERENCES restaurants (restaurant_id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS cart_items (
    cart_item_id TEXT PRIMARY KEY,
    user_id TEXT,
    item_id TEXT,
    quantity INTEGER,
    FOREIGN KEY (item_id) REFERENCES menu_items (item_id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS reviews (
    review_id TEXT PRIMARY KEY,
    restaurant_id TEXT,
    user_id TEXT,
    rating INTEGER,
    comment TEXT,
    review_time TEXT,
    FOREIGN KEY (restaurant_id) REFERENCES restaurants (restaurant_id)
)
""")

conn.commit()
print("New tables created (or already existed).")

In [ ]:
sample_restaurants = [
    ("REST01", "KFC", "Fast Food", 4.3, 30, "Model Town, Lahore"),
    ("REST02", "Pizza Hut", "Pizza", 4.1, 35, "DHA Phase 5, Lahore"),
    ("REST03", "McDonald's", "Fast Food", 4.5, 25, "Gulberg III, Lahore"),
    ("REST04", "Subway", "Sandwiches", 4.0, 20, "Johar Town, Lahore"),
    ("REST05", "Bundu Khan", "BBQ", 4.6, 40, "Gulberg II, Lahore"),
]

cursor.executemany(
    "INSERT OR IGNORE INTO restaurants VALUES (?, ?, ?, ?, ?, ?)",
    sample_restaurants
)
conn.commit()
print("Restaurants inserted.")

In [ ]:
sample_menu_items = [
    ("ITEM01", "REST01", "Zinger Burger", "Crispy chicken fillet burger", 650.0, "Burgers"),
    ("ITEM02", "REST01", "Fries", "Classic salted fries", 250.0, "Sides"),
    ("ITEM03", "REST02", "Large Pepperoni Pizza", "Loaded with pepperoni and cheese", 1899.0, "Pizza"),
    ("ITEM04", "REST02", "Garlic Bread", "Toasted garlic bread sticks", 400.0, "Sides"),
    ("ITEM05", "REST03", "Big Mac Meal", "Big Mac with fries and a drink", 990.0, "Meals"),
    ("ITEM06", "REST03", "McFlurry", "Oreo McFlurry dessert", 350.0, "Desserts"),
    ("ITEM07", "REST04", "Chicken Tikka Sub", "6-inch chicken tikka sandwich", 600.0, "Sandwiches"),
    ("ITEM08", "REST05", "Beef Seekh Kabab", "Grilled beef seekh kabab (6 pcs)", 1200.0, "BBQ"),
]

cursor.executemany(
    "INSERT OR IGNORE INTO menu_items VALUES (?, ?, ?, ?, ?, ?)",
    sample_menu_items
)
conn.commit()
print("Menu items inserted.")

In [ ]:
sample_reviews = [
    ("REV01", "REST01", "U001", 5, "Fast delivery and hot food!", "2026-09-10 14:00:00"),
    ("REV02", "REST01", "U002", 4, "Good but a bit pricey.", "2026-09-11 18:30:00"),
    ("REV03", "REST02", "U003", 3, "Pizza was cold on arrival.", "2026-09-12 20:15:00"),
    ("REV04", "REST03", "U001", 5, "Always consistent, love it.", "2026-09-13 12:45:00"),
    ("REV05", "REST05", "U004", 5, "Best BBQ in Lahore, hands down.", "2026-09-14 21:00:00"),
]

cursor.executemany(
    "INSERT OR IGNORE INTO reviews VALUES (?, ?, ?, ?, ?, ?)",
    sample_reviews
)
conn.commit()
print("Reviews inserted.")

In [ ]:
# Test
cursor.execute("SELECT * FROM restaurants")
for row in cursor.fetchall():
    print(row)

print()
cursor.execute("SELECT name, price FROM menu_items WHERE restaurant_id = 'REST01'")
for row in cursor.fetchall():
    print(row)

print()
cursor.execute("SELECT rating, comment FROM reviews WHERE restaurant_id = 'REST01'")
for row in cursor.fetchall():
    print(row)

In [ ]:
# Updated reset helper — covers ALL tables now
def reset_database():
    """Wipes and re-seeds ALL tables with fresh sample data."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("DELETE FROM orders")
    cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", sample_orders)

    cursor.execute("DELETE FROM restaurants")
    cursor.executemany("INSERT INTO restaurants VALUES (?, ?, ?, ?, ?, ?)", sample_restaurants)

    cursor.execute("DELETE FROM menu_items")
    cursor.executemany("INSERT INTO menu_items VALUES (?, ?, ?, ?, ?, ?)", sample_menu_items)

    cursor.execute("DELETE FROM cart_items")  # no sample data — cart starts empty

    cursor.execute("DELETE FROM reviews")
    cursor.executemany("INSERT INTO reviews VALUES (?, ?, ?, ?, ?, ?)", sample_reviews)

    conn.commit()
    conn.close()
    print("Database fully reset to original sample data.")

# Phase 17: Backend Functions for Restaurants, Menu, Cart & Reviews

In [ ]:
with open("db_functions.py", "a") as f:
    f.write('''

# ---------- Restaurants ----------

def get_all_restaurants():
    """Return a list of all restaurants as dictionaries."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM restaurants")
    rows = cursor.fetchall()
    conn.close()

    columns = ["restaurant_id", "name", "cuisine", "rating", "delivery_time_mins", "address"]
    return [dict(zip(columns, row)) for row in rows]


def search_restaurants(query=None, cuisine=None):
    """Search restaurants by name (partial match) and/or cuisine (exact match)."""
    conn = get_connection()
    cursor = conn.cursor()

    sql = "SELECT * FROM restaurants WHERE 1=1"
    params = []

    if query:
        sql += " AND name LIKE ?"
        params.append(f"%{query}%")

    if cuisine:
        sql += " AND cuisine = ?"
        params.append(cuisine)

    cursor.execute(sql, params)
    rows = cursor.fetchall()
    conn.close()

    columns = ["restaurant_id", "name", "cuisine", "rating", "delivery_time_mins", "address"]
    return [dict(zip(columns, row)) for row in rows]


def get_restaurant(restaurant_id):
    """Fetch a single restaurant by ID."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM restaurants WHERE restaurant_id = ?", (restaurant_id,))
    row = cursor.fetchone()
    conn.close()

    if row is None:
        return None

    columns = ["restaurant_id", "name", "cuisine", "rating", "delivery_time_mins", "address"]
    return dict(zip(columns, row))


# ---------- Menu ----------

def get_menu(restaurant_id):
    """Return all menu items for a given restaurant."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM menu_items WHERE restaurant_id = ?", (restaurant_id,))
    rows = cursor.fetchall()
    conn.close()

    columns = ["item_id", "restaurant_id", "name", "description", "price", "category"]
    return [dict(zip(columns, row)) for row in rows]


def get_menu_item(item_id):
    """Fetch a single menu item by ID."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM menu_items WHERE item_id = ?", (item_id,))
    row = cursor.fetchone()
    conn.close()

    if row is None:
        return None

    columns = ["item_id", "restaurant_id", "name", "description", "price", "category"]
    return dict(zip(columns, row))


# ---------- Cart ----------

def add_to_cart(user_id, item_id, quantity=1):
    """Add an item to a user's cart. If it's already there, increase quantity."""
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute(
        "SELECT cart_item_id, quantity FROM cart_items WHERE user_id = ? AND item_id = ?",
        (user_id, item_id)
    )
    existing = cursor.fetchone()

    if existing:
        new_quantity = existing[1] + quantity
        cursor.execute(
            "UPDATE cart_items SET quantity = ? WHERE cart_item_id = ?",
            (new_quantity, existing[0])
        )
    else:
        import uuid
        cart_item_id = f"CART{uuid.uuid4().hex[:8].upper()}"
        cursor.execute(
            "INSERT INTO cart_items VALUES (?, ?, ?, ?)",
            (cart_item_id, user_id, item_id, quantity)
        )

    conn.commit()
    conn.close()
    return True


def get_cart(user_id):
    """Return all cart items for a user, joined with menu item details."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT cart_items.cart_item_id, menu_items.name, menu_items.price,
               cart_items.quantity, menu_items.restaurant_id
        FROM cart_items
        JOIN menu_items ON cart_items.item_id = menu_items.item_id
        WHERE cart_items.user_id = ?
    """, (user_id,))
    rows = cursor.fetchall()
    conn.close()

    columns = ["cart_item_id", "name", "price", "quantity", "restaurant_id"]
    return [dict(zip(columns, row)) for row in rows]


def clear_cart(user_id):
    """Remove all items from a user's cart (used after placing an order)."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("DELETE FROM cart_items WHERE user_id = ?", (user_id,))
    conn.commit()
    conn.close()
    return True


# ---------- Placing an order ----------

def place_order(user_id, customer_name, restaurant_id, delivery_address):
    """Creates a new order from the user's current cart, then clears the cart."""
    cart = get_cart(user_id)
    if not cart:
        return None

    restaurant = get_restaurant(restaurant_id)
    if restaurant is None:
        return None

    items_summary = ", ".join(f"{c[\'quantity\']}x {c[\'name\']}" for c in cart)
    total_amount = sum(c["price"] * c["quantity"] for c in cart)

    import uuid
    from datetime import datetime
    order_id = f"ORD{uuid.uuid4().hex[:6].upper()}"
    order_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (order_id, customer_name, restaurant["name"], items_summary,
          "placed", delivery_address, None, "not_assigned",
          "cash_on_delivery", order_time, total_amount))
    conn.commit()
    conn.close()

    clear_cart(user_id)
    return order_id


# ---------- Reviews ----------

def get_reviews(restaurant_id):
    """Return all reviews for a restaurant."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM reviews WHERE restaurant_id = ?", (restaurant_id,))
    rows = cursor.fetchall()
    conn.close()

    columns = ["review_id", "restaurant_id", "user_id", "rating", "comment", "review_time"]
    return [dict(zip(columns, row)) for row in rows]


def add_review(restaurant_id, user_id, rating, comment):
    """Add a new review for a restaurant."""
    import uuid
    from datetime import datetime
    review_id = f"REV{uuid.uuid4().hex[:6].upper()}"
    review_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO reviews VALUES (?, ?, ?, ?, ?, ?)",
        (review_id, restaurant_id, user_id, rating, comment, review_time)
    )
    conn.commit()
    conn.close()
    return review_id
''')

print("Restaurant/menu/cart/review functions added to db_functions.py")

In [ ]:
# Reload — force fresh import
import sys
if "db_functions" in sys.modules:
    del sys.modules["db_functions"]
import db_functions

In [ ]:
# Test — full ordering flow
db_functions.add_to_cart("U005", "ITEM01", quantity=2)
db_functions.add_to_cart("U005", "ITEM02", quantity=1)

print("Cart:", db_functions.get_cart("U005"))

order_id = db_functions.place_order("U005", "Hina Tariq", "REST01", "House 22, Model Town, Lahore")
print("New order ID:", order_id)

print("Order details:", db_functions.get_order(order_id))
print("Cart after ordering:", db_functions.get_cart("U005"))

In [ ]:
# Test — search and reviews
print(db_functions.search_restaurants(cuisine="Fast Food"))
print(db_functions.get_reviews("REST01"))
db_functions.add_review("REST01", "U005", 5, "Loved the Zinger Burger!")
print(db_functions.get_reviews("REST01"))

# Phase 18: Streamlit UI

In [ ]:
# Install Streamlit
!pip install streamlit -q

In [ ]:
# Download cloudflared (recommended tunnel — more reliable than localtunnel for Streamlit)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

In [ ]:
%%writefile app.py

import streamlit as st
import db_functions

st.set_page_config(page_title="Foodpanda Demo", layout="wide")

USER_ID = "U005"

st.title("🍔 Foodpanda-Style Demo App")

tab1, tab2 = st.tabs(["Browse Restaurants", "Cart"])

with tab1:
    search_query = st.text_input("Search restaurants")
    cuisines = list(set(r["cuisine"] for r in db_functions.get_all_restaurants()))
    selected_cuisine = st.selectbox("Filter by cuisine", ["All"] + cuisines)

    cuisine_filter = None if selected_cuisine == "All" else selected_cuisine
    restaurants = db_functions.search_restaurants(query=search_query or None, cuisine=cuisine_filter)

    for r in restaurants:
        with st.expander(f"{r['name']} — {r['cuisine']} — ⭐ {r['rating']} — {r['delivery_time_mins']} min"):
            st.write(f"📍 {r['address']}")

            menu = db_functions.get_menu(r["restaurant_id"])
            for item in menu:
                col1, col2, col3 = st.columns([3, 1, 1])
                col1.write(f"**{item['name']}** — {item['description']}")
                col2.write(f"Rs. {item['price']}")
                if col3.button("Add to cart", key=f"add_{item['item_id']}"):
                    db_functions.add_to_cart(USER_ID, item["item_id"])
                    st.success(f"Added {item['name']} to cart!")

            st.subheader("Reviews")
            reviews = db_functions.get_reviews(r["restaurant_id"])
            for rev in reviews:
                st.write(f"⭐ {rev['rating']} — {rev['comment']}")

with tab2:
    st.subheader("Your Cart")
    cart = db_functions.get_cart(USER_ID)

    if not cart:
        st.info("Your cart is empty.")
    else:
        total = 0
        for item in cart:
            st.write(f"{item['quantity']}x {item['name']} — Rs. {item['price'] * item['quantity']}")
            total += item['price'] * item['quantity']

        st.write(f"**Total: Rs. {total}**")

        delivery_address = st.text_input("Delivery address", "House 1, Model Town, Lahore")
        customer_name = st.text_input("Your name", "Demo Customer")

        if st.button("Place Order"):
            restaurant_id = cart[0]["restaurant_id"]
            order_id = db_functions.place_order(USER_ID, customer_name, restaurant_id, delivery_address)
            if order_id:
                st.success(f"Order placed! Your order ID is {order_id}")
            else:
                st.error("Could not place order.")

In [ ]:
# Launch Streamlit in the background
!streamlit run app.py &>/content/logs.txt &
import time
time.sleep(3)
!cat /content/logs.txt

In [ ]:
# Confirm it is actually running before opening the tunnel
!ps aux | grep streamlit

In [ ]:
# Open the public tunnel (cloudflared) — copy the printed https://....trycloudflare.com link
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

# Phase 19: Order Tracking View

In [ ]:
%%writefile app.py

import streamlit as st
import db_functions

st.set_page_config(page_title="Foodpanda Demo", layout="wide")

USER_ID = "U005"

st.title("🍔 Foodpanda-Style Demo App")

tab1, tab2, tab3 = st.tabs(["Browse Restaurants", "Cart", "Track Order"])

with tab1:
    search_query = st.text_input("Search restaurants")
    cuisines = list(set(r["cuisine"] for r in db_functions.get_all_restaurants()))
    selected_cuisine = st.selectbox("Filter by cuisine", ["All"] + cuisines)

    cuisine_filter = None if selected_cuisine == "All" else selected_cuisine
    restaurants = db_functions.search_restaurants(query=search_query or None, cuisine=cuisine_filter)

    for r in restaurants:
        with st.expander(f"{r['name']} — {r['cuisine']} — ⭐ {r['rating']} — {r['delivery_time_mins']} min"):
            st.write(f"📍 {r['address']}")

            menu = db_functions.get_menu(r["restaurant_id"])
            for item in menu:
                col1, col2, col3 = st.columns([3, 1, 1])
                col1.write(f"**{item['name']}** — {item['description']}")
                col2.write(f"Rs. {item['price']}")
                if col3.button("Add to cart", key=f"add_{item['item_id']}"):
                    db_functions.add_to_cart(USER_ID, item["item_id"])
                    st.success(f"Added {item['name']} to cart!")

            st.subheader("Reviews")
            reviews = db_functions.get_reviews(r["restaurant_id"])
            for rev in reviews:
                st.write(f"⭐ {rev['rating']} — {rev['comment']}")

with tab2:
    st.subheader("Your Cart")
    cart = db_functions.get_cart(USER_ID)

    if not cart:
        st.info("Your cart is empty.")
    else:
        total = 0
        for item in cart:
            st.write(f"{item['quantity']}x {item['name']} — Rs. {item['price'] * item['quantity']}")
            total += item['price'] * item['quantity']

        st.write(f"**Total: Rs. {total}**")

        delivery_address = st.text_input("Delivery address", "House 1, Model Town, Lahore")
        customer_name = st.text_input("Your name", "Demo Customer")

        if st.button("Place Order"):
            restaurant_id = cart[0]["restaurant_id"]
            order_id = db_functions.place_order(USER_ID, customer_name, restaurant_id, delivery_address)
            if order_id:
                st.success(f"Order placed! Your order ID is {order_id}")
            else:
                st.error("Could not place order.")

with tab3:
    st.subheader("Track Your Order")
    order_id_input = st.text_input("Enter your Order ID", placeholder="e.g. ORD5216BD")

    if st.button("Track"):
        order = db_functions.get_order(order_id_input.strip())

        if order is None:
            st.error(f"No order found with ID {order_id_input}.")
        else:
            st.write(f"**Restaurant:** {order['restaurant']}")
            st.write(f"**Items:** {order['items']}")
            st.write(f"**Status:** {order['order_status'].replace('_', ' ').title()}")
            st.write(f"**Delivery Address:** {order['delivery_address']}")
            st.write(f"**Rider:** {order['rider_name'] or 'Not yet assigned'}")
            st.write(f"**Rider Status:** {order['rider_status'].replace('_', ' ').title()}")
            st.write(f"**Payment:** {order['payment_status'].replace('_', ' ').title()}")
            st.write(f"**Total:** Rs. {order['total_amount']}")

            status_steps = ["placed", "preparing", "out_for_delivery", "delivered"]
            if order["order_status"] in status_steps:
                current_step = status_steps.index(order["order_status"])
                st.progress((current_step + 1) / len(status_steps))
            elif order["order_status"] == "cancelled":
                st.warning("This order was cancelled.")

In [ ]:
# Restart Streamlit to pick up the change
!pkill -f streamlit
import time
time.sleep(2)
!streamlit run app.py &>/content/logs.txt &
time.sleep(3)
!cat /content/logs.txt

In [ ]:
# Verify app.py is complete (no truncated writefile) before opening the tunnel
with open("app.py", "r") as f:
    content = f.read()
print("File length:", len(content))
print("Contains status_steps line:", 'status_steps = ["placed", "preparing", "out_for_delivery", "delivered"]' in content)

In [ ]:
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

# Phase 20: Integrating the AI Support Chat

In [ ]:
# Confirm the API key is available before starting Streamlit
import os
assert "GOOGLE_API_KEY" in os.environ, "Run your Phase 5 API key cell first!"
print("API key is set.")

In [ ]:
%%writefile app.py

import streamlit as st
import db_functions
import graph as graph_module
from langchain_google_genai import ChatGoogleGenerativeAI
import os

st.set_page_config(page_title="Foodpanda Demo", layout="wide")

USER_ID = "U005"

st.title("🍔 Foodpanda-Style Demo App")

# --- Set up the AI agent once, cached across reruns ---
@st.cache_resource
def load_agent():
    llm = ChatGoogleGenerativeAI(
        model="gemini-flash-lite-latest",
        temperature=0
    )
    return graph_module.build_graph(llm)

def make_initial_state(user_message: str, history: str = "", user_id: str = "U001"):
    return {
        "user_message": user_message,
        "history": history,
        "user_id": user_id,
        "order_id": None,
        "order_status": None,
        "current_address": None,
        "action": None,
        "new_address": None,
        "rider_status": None,
        "result": None,
        "response": None,
    }

tab1, tab2, tab3, tab4 = st.tabs(["Browse Restaurants", "Cart", "Track Order", "AI Support"])

with tab1:
    search_query = st.text_input("Search restaurants")
    cuisines = list(set(r["cuisine"] for r in db_functions.get_all_restaurants()))
    selected_cuisine = st.selectbox("Filter by cuisine", ["All"] + cuisines)

    cuisine_filter = None if selected_cuisine == "All" else selected_cuisine
    restaurants = db_functions.search_restaurants(query=search_query or None, cuisine=cuisine_filter)

    for r in restaurants:
        with st.expander(f"{r['name']} — {r['cuisine']} — ⭐ {r['rating']} — {r['delivery_time_mins']} min"):
            st.write(f"📍 {r['address']}")

            menu = db_functions.get_menu(r["restaurant_id"])
            for item in menu:
                col1, col2, col3 = st.columns([3, 1, 1])
                col1.write(f"**{item['name']}** — {item['description']}")
                col2.write(f"Rs. {item['price']}")
                if col3.button("Add to cart", key=f"add_{item['item_id']}"):
                    db_functions.add_to_cart(USER_ID, item["item_id"])
                    st.success(f"Added {item['name']} to cart!")

            st.subheader("Reviews")
            reviews = db_functions.get_reviews(r["restaurant_id"])
            for rev in reviews:
                st.write(f"⭐ {rev['rating']} — {rev['comment']}")

with tab2:
    st.subheader("Your Cart")
    cart = db_functions.get_cart(USER_ID)

    if not cart:
        st.info("Your cart is empty.")
    else:
        total = 0
        for item in cart:
            st.write(f"{item['quantity']}x {item['name']} — Rs. {item['price'] * item['quantity']}")
            total += item['price'] * item['quantity']

        st.write(f"**Total: Rs. {total}**")

        delivery_address = st.text_input("Delivery address", "House 1, Model Town, Lahore")
        customer_name = st.text_input("Your name", "Demo Customer")

        if st.button("Place Order"):
            restaurant_id = cart[0]["restaurant_id"]
            order_id = db_functions.place_order(USER_ID, customer_name, restaurant_id, delivery_address)
            if order_id:
                st.success(f"Order placed! Your order ID is {order_id}")
            else:
                st.error("Could not place order.")

with tab3:
    st.subheader("Track Your Order")
    order_id_input = st.text_input("Enter your Order ID", placeholder="e.g. ORD5216BD")

    if st.button("Track"):
        order = db_functions.get_order(order_id_input.strip())

        if order is None:
            st.error(f"No order found with ID {order_id_input}.")
        else:
            st.write(f"**Restaurant:** {order['restaurant']}")
            st.write(f"**Items:** {order['items']}")
            st.write(f"**Status:** {order['order_status'].replace('_', ' ').title()}")
            st.write(f"**Delivery Address:** {order['delivery_address']}")
            st.write(f"**Rider:** {order['rider_name'] or 'Not yet assigned'}")
            st.write(f"**Rider Status:** {order['rider_status'].replace('_', ' ').title()}")
            st.write(f"**Payment:** {order['payment_status'].replace('_', ' ').title()}")
            st.write(f"**Total:** Rs. {order['total_amount']}")

            status_steps = ["placed", "preparing", "out_for_delivery", "delivered"]
            if order["order_status"] in status_steps:
                current_step = status_steps.index(order["order_status"])
                st.progress((current_step + 1) / len(status_steps))
            elif order["order_status"] == "cancelled":
                st.warning("This order was cancelled.")

with tab4:
    st.subheader("AI Support Chat")
    st.caption("Ask about cancelling an order, changing your address, or checking status.")

    if "chat_history" not in st.session_state:
        st.session_state.chat_history = []

    agent = load_agent()

    for role, message in st.session_state.chat_history:
        with st.chat_message(role):
            st.write(message)

    user_input = st.chat_input("Type your message...")

    if user_input:
        # Build the memory from earlier turns BEFORE adding the new message
        history = "\n".join(f"{role}: {msg}" for role, msg in st.session_state.chat_history[-8:])

        st.session_state.chat_history.append(("user", user_input))
        with st.chat_message("user"):
            st.write(user_input)

        final_state = agent.invoke(make_initial_state(user_input, history))
        response = final_state["response"]

        st.session_state.chat_history.append(("assistant", response))
        with st.chat_message("assistant"):
            st.write(response)

In [ ]:
# Restart Streamlit
!pkill -f streamlit
import time
time.sleep(2)
!streamlit run app.py &>/content/logs.txt &
time.sleep(3)
!cat /content/logs.txt

In [ ]:
# Verify app.py wrote completely before opening the tunnel
with open("app.py", "r") as f:
    content = f.read()
print("File length:", len(content))
print("Contains status_steps line:", 'status_steps = ["placed", "preparing", "out_for_delivery", "delivered"]' in content)
print("Contains AI Support tab:", 'AI Support' in content)

In [ ]:
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

# Phase 21: Policy Documents

Original policy documents used as the knowledge base for the RAG system.

In [ ]:
import os
os.makedirs("policies", exist_ok=True)

In [ ]:
%%writefile policies/refund_policy.txt

Refund Policy

If your order is cancelled before the restaurant starts preparing it, you will receive
a full refund within 3-5 business days to your original payment method.

If your order is cancelled after the restaurant has started preparing it but before
it is out for delivery, you will receive a 50% refund, since ingredients may have
already been used.

If an order arrives with missing items, you may request a partial refund for the
missing items only. Take a photo of what you received and contact support within
1 hour of delivery.

If an order arrives significantly late (more than 45 minutes past the estimated
delivery time), you are eligible for a delivery fee refund, even if you keep the order.

Refunds for cash-on-delivery orders are processed as store credit, not a cash refund,
since no online payment was made.

In [ ]:
%%writefile policies/cancellation_policy.txt

Cancellation Policy

Orders can be cancelled free of charge while their status is "placed" or "preparing."

Once an order's status changes to "out for delivery," it can no longer be cancelled,
since the rider has already picked it up and is on the way.

If you believe there was an error and want to cancel an order that is already out for
delivery, you may refuse the delivery when the rider arrives. This does not guarantee
a refund and is handled on a case-by-case basis.

Repeated cancellations (more than 5 in a 30-day period) may result in a temporary
restriction on cash-on-delivery orders for your account.

In [ ]:
%%writefile policies/delivery_delay_policy.txt

Delivery Delay Policy

Estimated delivery times shown in the app are approximate and can vary due to weather,
traffic, or high order volume during peak hours (lunch: 12-2 PM, dinner: 7-10 PM).

If your order is more than 30 minutes late, you will receive a discount code for 15%
off your next order, automatically applied to your account.

If your order is more than 45 minutes late, you are eligible for a delivery fee refund
on the current order — see the Refund Policy for details.

If your order does not arrive at all within 2 hours of the estimated delivery time,
contact support immediately for a full refund and priority handling.

In [ ]:
%%writefile policies/tipping_policy.txt

Rider Tipping Policy

Tipping your rider is optional and greatly appreciated, especially during bad weather
or for long-distance deliveries.

You can tip in cash directly to the rider upon delivery, or add a tip through the app
before checkout, which is included in your total and paid to the rider automatically.

100% of tips go directly to the rider — the platform does not take any commission on
tips.

If you tipped through the app but the rider did not receive it (e.g. order was
cancelled after tipping), the tip amount will be automatically refunded to you.

In [ ]:
# Confirm all files were created correctly
import os
for filename in os.listdir("policies"):
    path = os.path.join("policies", filename)
    with open(path) as f:
        content = f.read()
    print(f"{filename}: {len(content)} characters")

In [ ]:
# Test
with open("policies/refund_policy.txt") as f:
    print(f.read())

# Phase 22: Chunking & Embedding the Policy Documents

Uses `chromadb` directly with its bundled local embedding model.

In [ ]:
# Install chromadb + pin opentelemetry to a working version.
# (Plain `pip install chromadb` pulls opentelemetry 1.45.0, which is broken:
#  "cannot import name OTEL_LOGRECORD_ATTRIBUTE_COUNT_LIMIT".)
!pip freeze | grep -i opentelemetry | cut -d= -f1 | xargs -r pip uninstall -y -q
!pip install -q chromadb
!pip install -q "opentelemetry-api==1.27.0" "opentelemetry-sdk==1.27.0" "opentelemetry-semantic-conventions==0.48b0" "opentelemetry-exporter-otlp-proto-grpc==1.27.0" "opentelemetry-exporter-otlp-proto-common==1.27.0" "opentelemetry-proto==1.27.0"

# Check in a brand-new Python process (ignores anything already loaded in this session)
!python -c "import chromadb; print('chromadb import OK, version', chromadb.__version__)"
print()
print("If the line above says 'import OK' but the next cell still fails,")
print("do Runtime -> Restart session, then Run all again (this cell re-pins automatically).")


In [ ]:
# Build the vector store — persisted to your Drive folder so it survives runtime restarts
import chromadb
import os

chroma_client = chromadb.PersistentClient(path="chroma_db")
policy_collection = chroma_client.get_or_create_collection(name="policies")


def chunk_text(text, min_length=20):
    """Splits a document into paragraphs. The first paragraph is the document title
    (e.g. 'Refund Policy'); it is not its own chunk, but it is added in front of every
    other paragraph so each chunk still knows which policy it belongs to."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    if not paragraphs:
        return []
    title, body = paragraphs[0], paragraphs[1:]
    return [f"{title}: {p}" for p in body if len(p) > min_length]


policy_files = os.listdir("policies")
all_chunks = []
all_ids = []
all_metadata = []

for filename in policy_files:
    path = os.path.join("policies", filename)
    with open(path) as f:
        text = f.read()
    chunks = chunk_text(text)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_ids.append(f"{filename}_{i}")
        all_metadata.append({"source": filename})

# Clear any previous run's data first, so re-running this cell doesn't duplicate chunks
existing_ids = policy_collection.get()["ids"]
if existing_ids:
    policy_collection.delete(ids=existing_ids)

policy_collection.add(
    documents=all_chunks,
    ids=all_ids,
    metadatas=all_metadata
)

print(f"Added {len(all_chunks)} chunks from {len(policy_files)} policy documents.")

In [ ]:
# A retriever function — searches by MEANING, not exact keyword match
def search_policies(query, k=3):
    results = policy_collection.query(query_texts=[query], n_results=k)
    chunks = results["documents"][0]
    sources = [m["source"] for m in results["metadatas"][0]]
    return list(zip(chunks, sources))

In [ ]:
# Test
results = search_policies("What happens if my order is late?")
for chunk, source in results:
    print(f"[{source}]")
    print(chunk)
    print()

In [ ]:
# Test 2 — a differently-worded question that should still find the tipping policy
results2 = search_policies("Do I have to tip the delivery guy?")
for chunk, source in results2:
    print(f"[{source}]")
    print(chunk)
    print()

### Tuning the retrieval cutoff

Smaller distance = closer in meaning. `search_policy` ignores chunks with distance >= `MAX_DISTANCE` (default `1.2`).

In [ ]:
# Tuning cell - shows how close each question is to the best-matching policy chunk.
test_questions = [
    ("policy", "What happens if my order is late?"),
    ("policy", "Do I have to tip the delivery guy?"),
    ("policy", "How much refund do I get if I cancel?"),
    ("policy", "Can I cancel after the rider picks it up?"),
    ("off-topic", "What toppings does Pizza Hut have?"),
    ("off-topic", "defense house 12 lahore"),
    ("off-topic", "What is the capital of France?"),
]
for kind, q in test_questions:
    res = policy_collection.query(query_texts=[q], n_results=1, include=["distances"])
    print(f"{kind:9} best distance = {res['distances'][0][0]:.2f}   |   {q}")


### How it works

- **Chunking**: we split each policy document into paragraphs rather than searching whole files. This matters because a question usually matches one specific paragraph, not an entire document — smaller chunks give more precise retrieval.
- **Embedding**: converting text into a list of numbers (a vector) that captures its *meaning*. `chromadb`'s default embedding model does this automatically when you call `.add()` and `.query()` — you never see the numbers directly, but it's what allows Test 2 to find the tipping policy even though the words "tip the delivery guy" don't literally appear in the document.
- **Vector store**: a database (here, `chromadb`) built specifically to store embeddings and quickly find the most similar ones to a new query — this is the "search by meaning" engine behind RAG.
- **`PersistentClient(path="chroma_db")`**: saves the vector store as files in your Drive folder, so you don't have to re-embed everything each session.

# Phase 23: RAG Tool + Routing

Adds a `search_policy` tool, a node that answers general questions from retrieved policy text, and a conditional edge that routes order-specific requests and general questions separately.

In [ ]:
# Add the search_policy tool to tools.py
with open("tools.py", "a") as f:
    f.write('''

import chromadb

_chroma_client = chromadb.PersistentClient(path="chroma_db")
_policy_collection = _chroma_client.get_or_create_collection(name="policies")

# Lower = stricter. Use the tuning cell in Phase 22 to pick a good value.
MAX_DISTANCE = 1.2


@tool
def search_policy(query: str) -> str:
    """Search company policy documents (refunds, cancellations, delivery delays,
    tipping) for information relevant to a general question that is not about
    a specific order."""
    results = _policy_collection.query(
        query_texts=[query], n_results=3, include=["documents", "distances"]
    )
    # Keep only chunks that are actually close in meaning to the question
    chunks = [
        doc for doc, dist in zip(results["documents"][0], results["distances"][0])
        if dist < MAX_DISTANCE
    ]
    if not chunks:
        return "No relevant policy information found."
    return "\\n\\n".join(chunks)
''')
print("search_policy tool added to tools.py")

In [ ]:
%%writefile nodes.py

import json
from langchain_core.messages import HumanMessage
from state import SupportState
import db_functions
import tools as tools_module


def get_text(response):
    """Handles both plain-string and structured (list-of-blocks) Gemini responses."""
    if isinstance(response.content, str):
        return response.content
    return "".join(
        block.get("text", "")
        for block in response.content
        if isinstance(block, dict)
    )


def make_understand_request_node(llm):
    """Returns a node function that uses the LLM to figure out the user's
    intent (action), which order they mean (order_id), and — for address
    changes — the new address they want."""

    def understand_request(state: SupportState) -> dict:
        prompt = f"""You are analyzing a customer support message for a food delivery app.

Conversation so far (may be empty):
{state.get('history') or '(none)'}

Latest customer message: "{state['user_message']}"

IMPORTANT: the latest message is often an answer to something the assistant just asked
(for example only an order ID, or only a new address). Use the conversation so far to work
out what the customer really wants, and carry over the order_id and new_address from earlier
messages when the latest message does not repeat them. If the latest message is only an order ID
or only an address, keep the earlier intent (cancel / change_address / check_status) - do NOT
answer with "other" or "policy_question".
If the customer asks for two things in one message (for example "cancel it, and if you can't,
change the address"), set "action" to the FIRST thing they ask for.

Figure out three things:
1. "action": one of exactly these values: "cancel", "change_address", "check_status", "policy_question", "other"
   - Use "policy_question" for general questions about refunds, cancellation rules, delivery
     delay compensation, tipping, or any question that is NOT about a specific existing order.
2. "order_id": the order ID mentioned in the message (like "ORD1002"), or null if none is mentioned
3. "new_address": if the user gave a new delivery address, put it here as a string; otherwise null

Reply with ONLY a JSON object, nothing else, like this:
{{"action": "change_address", "order_id": "ORD1002", "new_address": "House 5, Gulberg, Lahore"}}
"""
        response = llm.invoke([HumanMessage(content=prompt)])
        text = get_text(response).strip()

        if text.startswith("```"):
            text = text.strip("`")
            text = text.replace("json", "", 1).strip()

        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            parsed = {"action": "other", "order_id": None, "new_address": None}

        return {
            "action": parsed.get("action"),
            "order_id": parsed.get("order_id"),
            "new_address": parsed.get("new_address"),
        }

    return understand_request


def find_order_node(state: SupportState) -> dict:
    """Looks up the order in the database. Pure database lookup — no LLM."""
    order_id = state.get("order_id")

    if not order_id:
        return {"result": "no_order_id"}

    order = db_functions.get_order(order_id)

    if order is None:
        return {"result": "order_not_found"}

    return {
        "order_status": order["order_status"],
        "current_address": order["delivery_address"],
        "rider_status": order["rider_status"],
    }


# Business rules — plain Python, no LLM involved in deciding these
CANCELLABLE_STATUSES = ["placed", "preparing"]
ADDRESS_CHANGEABLE_STATUSES = ["placed", "preparing"]


def decide_action(state: SupportState) -> dict:
    """Pure business-rule decision — NO LLM involved. Looks at the real
    order_status and the requested action, and decides the outcome."""
    action = state.get("action")
    order_status = state.get("order_status")

    if state.get("result") in ("no_order_id", "order_not_found"):
        return {}  # already handled — nothing to decide

    if order_status == "cancelled" and action in ("cancel", "change_address"):
        return {"result": "already_cancelled"}

    if action == "cancel":
        if order_status in CANCELLABLE_STATUSES:
            success = db_functions.update_order_status(state["order_id"], "cancelled")
            return {"result": "cancelled" if success else "cancel_failed"}
        return {"result": "cancel_blocked"}

    if action == "change_address":
        if order_status not in ADDRESS_CHANGEABLE_STATUSES:
            return {"result": "address_blocked"}
        new_address = state.get("new_address")
        if not new_address:
            return {"result": "address_missing"}
        success = db_functions.update_delivery_address(state["order_id"], new_address)
        return {"result": "address_updated" if success else "address_update_failed"}

    if action == "check_status":
        return {"result": "status_checked"}

    return {"result": "unrecognized_action"}


RESULT_MEANINGS = {
    "no_order_id": "The customer has not given an order ID yet. Ask for it.",
    "order_not_found": "No order exists with that ID. Ask them to double-check the ID.",
    "cancelled": "The order was successfully cancelled. Refund details (if any) follow in the result text.",
    "cancel_blocked": "The order could NOT be cancelled because it is already out for delivery or delivered. Orders can only be cancelled while 'placed' or 'preparing'.",
    "cancel_failed": "Something went wrong while cancelling. Apologise and suggest trying again.",
    "address_updated": "The delivery address was successfully updated to the new address.",
    "address_missing": "The customer wants to change the address but has not said the new address yet. Ask for it.",
    "address_blocked": "The address could NOT be changed because the order is already out for delivery or delivered. Addresses can only be changed while the order is 'placed' or 'preparing'.",
    "address_update_failed": "Something went wrong while updating the address. Apologise and suggest trying again.",
    "status_checked": "Tell the customer the current order status, and the rider information if it is given.",
    "already_cancelled": "This order was ALREADY cancelled earlier, so it cannot be changed or cancelled again, and a cancelled order cannot be reactivated. If they still want food, they need to place a new order.",
}


def explain_result(result):
    """Turns a result code into a plain-English meaning, so the LLM never has to guess."""
    result = result or ""
    for key in sorted(RESULT_MEANINGS, key=len, reverse=True):
        if result.startswith(key):
            return RESULT_MEANINGS[key]
    return "No special outcome."


def make_respond_node(llm):
    """Returns a node function that turns the final state into a friendly,
    natural-language response — but only DESCRIBES the result, never decides it."""

    def respond(state: SupportState) -> dict:
        prompt = f"""You are a friendly food delivery support assistant.

Here is what happened when handling the customer's request:
- Customer said: "{state['user_message']}"
- Order ID: {state.get('order_id')}
- Order status: {state.get('order_status')}
- Result: {state.get('result')}
- What this result means: {explain_result(state.get('result'))}
- Rider info: {state.get('rider_status')}

Conversation so far (may be empty):
{state.get('history') or '(none)'}

Write a short, friendly response to the customer explaining the outcome.
If the result is "address_missing", politely ask them what new address they'd like.
If the result is "cancel_blocked" and the customer also asked to change the address instead, tell them the address cannot be changed at this stage either, because the same rule applies (only orders that are "placed" or "preparing" can be changed).
Base your answer ONLY on the result and its meaning above. Never invent a reason that is not stated there.
If there is earlier conversation, do NOT start with a greeting like "Hi there" again.
Do not invent any information that isn't given above.
"""
        response = llm.invoke([HumanMessage(content=prompt)])
        return {"response": get_text(response)}

    return respond


def gather_extra_info(state: SupportState) -> dict:
    """Deterministically calls LangChain tools based on the already-decided
    result — NOT based on LLM judgment. This is where LangChain tools and
    LangGraph flow control meet (Phase 12)."""
    result = state.get("result")
    order_id = state.get("order_id")
    updates = {}

    if result == "cancelled":
        refund_info = tools_module.check_refund.invoke(
            {"order_id": order_id, "previous_status": state.get("order_status") or ""}
        )
        updates["result"] = f"cancelled. {refund_info}"

    elif result == "status_checked":
        rider_info = tools_module.get_rider_status.invoke({"order_id": order_id})
        updates["rider_status"] = rider_info

    return updates


def make_answer_policy_question_node(llm):
    """Returns a node function that answers general/policy questions using RAG:
    it retrieves relevant policy text via the search_policy tool, then asks the
    LLM to answer USING ONLY that retrieved text — never inventing information."""

    def answer_policy_question(state: SupportState) -> dict:
        context_chunks = tools_module.search_policy.invoke({"query": state["user_message"]})

        prompt = f"""You are a friendly food delivery support assistant.

A customer asked: "{state['user_message']}"

Here is relevant policy information found for this question:
{context_chunks}

Using ONLY the information above, write a short, friendly answer.
If the information above does not actually answer the question, politely say
you don't have information on that specific topic and suggest contacting
support for more help. Do not invent any information not present above.
"""
        response = llm.invoke([HumanMessage(content=prompt)])
        return {"result": "policy_answered", "response": get_text(response)}

    return answer_policy_question

In [ ]:
%%writefile graph.py

from langgraph.graph import StateGraph, END
from state import SupportState
import nodes


def route_after_understanding(state: SupportState) -> str:
    """Conditional edge — sends order-specific requests down the existing
    find_order path, and everything else (policy questions, unclear
    requests) to the RAG-based policy-answering path instead."""
    if state.get("action") in ("cancel", "change_address", "check_status"):
        return "find_order"
    return "answer_policy_question"


def route_after_find_order(state: SupportState) -> str:
    """Conditional edge function. Looks at state, returns the NAME of the
    next node to go to. Runs after find_order."""
    if state.get("result") in ("no_order_id", "order_not_found"):
        return "respond"
    return "decide_action"


def build_graph(llm):
    understand_request = nodes.make_understand_request_node(llm)
    respond = nodes.make_respond_node(llm)
    answer_policy_question = nodes.make_answer_policy_question_node(llm)

    graph_builder = StateGraph(SupportState)

    graph_builder.add_node("understand_request", understand_request)
    graph_builder.add_node("find_order", nodes.find_order_node)
    graph_builder.add_node("decide_action", nodes.decide_action)
    graph_builder.add_node("gather_extra_info", nodes.gather_extra_info)
    graph_builder.add_node("respond", respond)
    graph_builder.add_node("answer_policy_question", answer_policy_question)

    graph_builder.set_entry_point("understand_request")

    graph_builder.add_conditional_edges(
        "understand_request",
        route_after_understanding,
        {
            "find_order": "find_order",
            "answer_policy_question": "answer_policy_question",
        }
    )

    graph_builder.add_conditional_edges(
        "find_order",
        route_after_find_order,
        {
            "respond": "respond",
            "decide_action": "decide_action",
        }
    )

    graph_builder.add_edge("decide_action", "gather_extra_info")
    graph_builder.add_edge("gather_extra_info", "respond")
    graph_builder.add_edge("respond", END)
    graph_builder.add_edge("answer_policy_question", END)

    return graph_builder.compile()

In [ ]:
# Rebuild the app — force a fresh import of every module
import sys
for mod_name in ["db_functions", "tools", "state", "nodes", "graph"]:
    if mod_name in sys.modules:
        del sys.modules[mod_name]

import db_functions
import tools
import state
from state import SupportState
import nodes
import graph as graph_module

app = graph_module.build_graph(llm)
print("Graph rebuilt successfully with RAG routing.")

In [ ]:
# Test — the exact case that failed in Phase 14
r = app.invoke(make_initial_state("What toppings does Pizza Hut have?"))
print("Result:", r["result"])
print("Response:", r["response"])

In [ ]:
# Test — a real policy question, answered from the RAG knowledge base
r2 = app.invoke(make_initial_state("What happens if my order is really late?"))
print("Result:", r2["result"])
print("Response:", r2["response"])

In [ ]:
# Test — another policy question, differently worded
r3 = app.invoke(make_initial_state("Do I need to tip my delivery rider?"))
print("Result:", r3["result"])
print("Response:", r3["response"])

In [ ]:
# Regression test — confirm order-specific flows still work correctly
reset_database()

r4 = app.invoke(make_initial_state("Cancel my order ORD1001"))
print("Cancel test:", r4["result"], "-", r4["response"][:80])

r5 = app.invoke(make_initial_state("Where is my order ORD1002?"))
print("Status test:", r5["result"], "-", r5["response"][:80])

In [ ]:
# Updated full test suite — includes the RAG fix
test_cases_v2 = test_cases + [
    {"label": "Policy question — delivery delay", "message": "What happens if my order is really late?", "expected_result": "policy_answered"},
    {"label": "Policy question — tipping", "message": "Do I need to tip my delivery rider?", "expected_result": "policy_answered"},
    {"label": "Unrelated menu question (Phase 14 fix)", "message": "What toppings does Pizza Hut have?", "expected_result": "policy_answered"},
]

reset_database()
run_test_suite(test_cases_v2)

# Phase 24: Multi-turn Conversation Tests

Tests keep chat history between messages, like the Streamlit chat does.

In [ ]:
def build_history(turns, max_turns=8):
    return "\n".join(f"{role}: {msg}" for role, msg in turns[-max_turns:])


def run_conversation(label, steps):
    """steps = list of (user_message, expected_result_prefix). Keeps the chat history
    between turns, like the Streamlit chat does. Returns True if every turn passed."""
    reset_database()
    turns = []
    all_ok = True
    print(f"=== {label} ===")
    for message, expected in steps:
        state = app.invoke(make_initial_state(message, history=build_history(turns)))
        actual = state.get("result") or ""
        ok = actual.startswith(expected)
        all_ok = all_ok and ok
        print(f"[{'PASS' if ok else 'FAIL'}] You: {message}")
        print(f"       expected {expected!r}, got {actual!r}")
        print(f"       Agent: {state['response']}\n")
        turns.append(("user", message))
        turns.append(("assistant", state["response"]))
    return all_ok


In [ ]:
conversation_results = [
    run_conversation("Address change across several messages", [
        ("I want to change my delivery address", "no_order_id"),
        ("ORD1004 is my order id", "address_missing"),
        ("House 12, Defense, Lahore", "address_updated"),
    ]),
    run_conversation("Cancel across several messages", [
        ("i want to cancel the order", "no_order_id"),
        ("ORD1001 this is the order id", "cancelled"),
    ]),
    run_conversation("Cancel + fallback address in one message (cancel is blocked)", [
        ("cancel order ORD1002, and if you can't cancel it please change the address to House 5, DHA, Lahore",
         "cancel_blocked"),
    ]),
    run_conversation("Change address AFTER the order was already cancelled", [
        ("i want to cancel my order ORD1004", "cancelled"),
        ("can i change the address instead of cancel the order", "already_cancelled"),
        ("defense 2345 house no 56", "already_cancelled"),
    ]),
    run_conversation("Your original screenshot conversation", [
        ("i want to cancel the order", "no_order_id"),
        ("ORD1004 this is my order number, if the order can't be cancelled can you please change the address",
         "cancelled"),
    ]),
]
print("ALL CONVERSATIONS PASSED" if all(conversation_results)
      else "SOME CONVERSATIONS FAILED - read the output above")


# Phase 25: Launch the Demo App

In [ ]:
import os, time, subprocess, urllib.request

# ---- settings ----
RESET_DEMO_DATA = True   # True = start the demo with fresh sample data (wipes orders made in earlier tests)
PROJECT_PATH = "/content/drive/MyDrive/foodpanda-ai-agent"
# ------------------

os.chdir(PROJECT_PATH)

# 1) Make sure everything the app needs exists
assert "GOOGLE_API_KEY" in os.environ, "Run the Phase 5 API key cell first!"
for f in ["db_functions.py", "tools.py", "state.py", "nodes.py", "graph.py", "app.py", "foodpanda.db", "chroma_db"]:
    assert os.path.exists(f), f"Missing '{f}' - run the earlier phases first."

app_code = open("app.py").read()
assert "AI Support" in app_code and "history" in app_code, (
    "app.py is not the final version - re-run the Phase 20 '%%writefile app.py' cell "
    "(and do NOT run the Phase 18/19 app.py cells after it).")

# 2) Optional clean start
if RESET_DEMO_DATA:
    try:
        reset_database()
    except NameError:
        print("reset_database() is not defined in this session - skipping reset.")

# 3) (Re)start Streamlit in the background
subprocess.run(["pkill", "-f", "streamlit run"])
time.sleep(2)
log = open("/content/logs.txt", "w")
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=log, stderr=subprocess.STDOUT,
)

# 4) Wait until it answers
for _ in range(40):
    try:
        urllib.request.urlopen("http://localhost:8501", timeout=2)
        print("Streamlit is running on port 8501. Now run the next cell to open the public link.")
        break
    except Exception:
        time.sleep(1)
else:
    print("Streamlit did not start. Logs:\n")
    print(open("/content/logs.txt").read())

In [ ]:
import os, subprocess

os.chdir("/content/drive/MyDrive/foodpanda-ai-agent")

if not os.path.exists("cloudflared-linux-amd64"):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod("cloudflared-linux-amd64", 0o755)

print("Look for the https://....trycloudflare.com link in the output below.")
print("KEEP THIS CELL RUNNING while you use the app - stopping it kills the link.\n")

!./cloudflared-linux-amd64 tunnel --url http://localhost:8501